# LeetCode 1208 — Get Equal Substrings Within Budget bằng Python

Link đề: https://leetcode.com/problems/get-equal-substrings-within-budget/ · Độ khó: **Medium** · Chủ đề: String, Binary Search, Sliding Window, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def equalSubstring(self, s: str, t: str, maxCost: int) -> int:
        ...
```

Ghi chú: biến đổi về bài toán số: `cost[i] = |ord(s[i]) − ord(t[i])| ≥ 0`. Đề trở thành **"đoạn con dài nhất có tổng ≤ maxCost"** trên mảng **không âm** — bài mẫu kinh điển của **cửa sổ trượt (sliding window)**. Cũng giải được bằng **tổng tiền tố + tìm kiếm nhị phân**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Biến đổi đề chuỗi thành mảng chi phí `cost[i] = |s[i] − t[i]|` (dùng `ord`).
2. Nhận ra vì sao "mảng **không âm**" là điều kiện để cửa sổ trượt hoạt động: mở rộng làm tổng **không giảm**, thu hẹp làm tổng **không tăng**.
3. Cài **cửa sổ trượt co giãn** (`while sum > maxCost: left += 1`) và biến thể **cửa sổ không bao giờ co** (chỉ trượt).
4. Dùng **tổng tiền tố** + **tìm kiếm nhị phân** (theo độ dài hoặc theo đầu trái) như lời giải thay thế `O(n log n)`.
5. Tránh bẫy: dùng `if` thay `while` khi thu hẹp, tính độ dài lệch 1 (`right − left` vs `right − left + 1`), áp dụng cửa sổ trượt cho mảng có số âm.

## 2. Đề bài & ví dụ

Cho hai chuỗi `s` và `t` **cùng độ dài** và số nguyên `maxCost`. Đổi ký tự thứ `i` của `s` thành ký tự thứ `i` của `t` tốn `|s[i] − t[i]|` (hiệu giá trị ASCII).

Trả về **độ dài lớn nhất** của một chuỗi con (liên tiếp) của `s` có thể đổi thành chuỗi con tương ứng (cùng vị trí) của `t` với tổng chi phí **≤ maxCost**. Nếu không có, trả về `0`.

**Ví dụ 1:** `s = "abcd"`, `t = "bcdf"`, `maxCost = 3` → `3` (`"abc"` → `"bcd"` tốn `1+1+1 = 3`).

**Ví dụ 2:** `s = "abcd"`, `t = "cdef"`, `maxCost = 3` → `1` (mỗi ký tự tốn `2`).

**Ví dụ 3:** `s = "abcd"`, `t = "acde"`, `maxCost = 0` → `1` (chỉ `"a"` không tốn gì).

**Ràng buộc:** `1 <= s.length <= 10^5`; `t.length == s.length`; `0 <= maxCost <= 10^6`; `s`, `t` chỉ gồm chữ thường.

In [1]:
import bisect
import random
import time
from itertools import accumulate
from typing import List

examples = [
    ("abcd", "bcdf", 3, 3),
    ("abcd", "cdef", 3, 1),
    ("abcd", "acde", 0, 1),
]
for s, t, mc, want in examples:
    print(f"s={s!r} t={t!r} maxCost={mc} -> {want}")

s='abcd' t='bcdf' maxCost=3 -> 3
s='abcd' t='cdef' maxCost=3 -> 1
s='abcd' t='acde' maxCost=0 -> 1


## 3. Kiến thức nền

### 3.1 `ord` và chi phí từng vị trí

`ord(ch)` trả mã Unicode/ASCII của ký tự (`ord('a') = 97`). Chi phí vị trí `i`: `abs(ord(s[i]) - ord(t[i]))`. Dùng `zip(s, t)` để duyệt song song.

In [2]:
s, t = "abcd", "bcdf"
cost = [abs(ord(a) - ord(b)) for a, b in zip(s, t)]
print(ord('a'), ord('f'), cost)

97 102 [1, 1, 1, 2]


### 3.2 Tổng tiền tố (prefix sum)

`P[0] = 0`, `P[i+1] = P[i] + cost[i]`. Tổng đoạn `cost[l..r] = P[r+1] − P[l]` trong `O(1)`. Vì `cost ≥ 0`, mảng `P` **không giảm** — điều kiện để tìm kiếm nhị phân trên `P`.

In [3]:
P = list(accumulate(cost, initial=0))
print(P, "tổng cost[0..2] =", P[3] - P[0])

[0, 1, 2, 3, 5] tổng cost[0..2] = 3


### 3.3 Hai con trỏ / cửa sổ trượt

Cửa sổ là đoạn `[left, right]`. Mỗi bước tăng `right` (mở rộng), cập nhật tổng; nếu tổng vượt ngưỡng thì tăng `left` (thu hẹp) cho tới khi hợp lệ. Mỗi chỉ số vào và ra cửa sổ tối đa một lần → `O(n)`.

In [4]:
arr, limit = [1, 2, 1, 0, 3], 3
left = total = 0
for right, x in enumerate(arr):
    total += x
    while total > limit:
        total -= arr[left]; left += 1
    print(f"right={right} cửa sổ=[{left},{right}] tổng={total} dài={right - left + 1}")

right=0 cửa sổ=[0,0] tổng=1 dài=1
right=1 cửa sổ=[0,1] tổng=3 dài=2
right=2 cửa sổ=[1,2] tổng=3 dài=2
right=3 cửa sổ=[1,3] tổng=3 dài=3
right=4 cửa sổ=[3,4] tổng=3 dài=2


### 3.4 `bisect_left` trên mảng không giảm

`bisect_left(P, v)` = vị trí đầu tiên `P[p] >= v`. Với đầu phải `r` cố định, đầu trái nhỏ nhất `l` thỏa `P[r+1] − P[l] ≤ maxCost` ⇔ `P[l] ≥ P[r+1] − maxCost` ⇒ `l = bisect_left(P, P[r+1] − maxCost)`.

In [5]:
P = [0, 1, 2, 3, 5]
print(bisect.bisect_left(P, 2), bisect.bisect_left(P, 2.5), bisect.bisect_left(P, -1))

2 3 0


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1: `cost = [1, 1, 1, 2]`, `maxCost = 3`. Liệt kê các đoạn:

| đoạn | tổng | ≤ 3? | dài |
|---|---|---|---|
| [0..2] | 3 | ✓ | **3** |
| [1..3] | 4 | ✗ | |
| [0..3] | 5 | ✗ | |
| [2..3] | 3 | ✓ | 2 |

Quan sát quan trọng: nếu đoạn `[l..r]` **quá đắt** thì mọi đoạn **chứa** nó (`[l'..r']` với `l' ≤ l`, `r' ≥ r`) cũng quá đắt (vì chi phí không âm). Ngược lại, nếu `[l..r]` hợp lệ thì mọi đoạn con của nó cũng hợp lệ. Tính chất "đơn điệu" này là chìa khóa.

In [6]:
cost = [1, 1, 1, 2]
for l in range(4):
    print(l, [(r, sum(cost[l:r + 1])) for r in range(l, 4)])

0 [(0, 1), (1, 2), (2, 3), (3, 5)]
1 [(1, 1), (2, 2), (3, 4)]
2 [(2, 1), (3, 3)]
3 [(3, 2)]


## 5. Cách 1 — Brute Force / baseline

Duyệt mọi đầu trái `l`, mở rộng `r` và cộng dồn; dừng khi vượt ngân sách (vì chi phí không âm, mở rộng thêm chỉ đắt hơn). Ghi nhận độ dài lớn nhất.

In [7]:
def brute(s: str, t: str, maxCost: int) -> int:
    n = len(s)
    best = 0
    for l in range(n):
        total = 0
        for r in range(l, n):
            total += abs(ord(s[r]) - ord(t[r]))
            if total > maxCost:
                break
            best = max(best, r - l + 1)
    return best


def brute_raw(s, t, maxCost):
    """Oracle O(n^3): thử mọi đoạn, tính tổng từ đầu."""
    n, best = len(s), 0
    for l in range(n):
        for r in range(l, n):
            if sum(abs(ord(s[i]) - ord(t[i])) for i in range(l, r + 1)) <= maxCost:
                best = max(best, r - l + 1)
    return best


for s, t, mc, want in examples:
    print(brute(s, t, mc), brute_raw(s, t, mc), want)
    assert brute(s, t, mc) == brute_raw(s, t, mc) == want

3 3 3
1 1 1
1 1 1


**Độ phức tạp:** worst case (ví dụ `s == t`, mọi đoạn đều hợp lệ, không `break` được) là `O(n^2)` → `5·10^9` bước với `n = 10^5` → **TLE**.

**Nút thắt:** với mỗi `l` ta tính lại từ đầu phần tổng đã biết ở `l − 1`. Khi chuyển từ `l` sang `l + 1`, đầu phải tốt nhất **không lùi** — không cần quay về `r = l`.

In [8]:
for n in [1000, 2000, 4000]:
    s = 'a' * n
    t0 = time.perf_counter(); brute(s, s, 0); print(n, f"{time.perf_counter() - t0:.3f}s")

1000 0.050s


2000 0.205s


4000 0.831s


## 6. Bước đột phá

**Insight:** gọi `R(l)` = đầu phải xa nhất sao cho `[l..R(l)]` hợp lệ. Vì chi phí không âm, `R(l)` **không giảm** theo `l` (bỏ bớt phần tử đầu thì tổng không tăng, nên đi được ít nhất xa như cũ). ⇒ Hai con trỏ chỉ tiến, không lùi.

Phát biểu theo đầu phải (dễ cài hơn): với mỗi `right`, giữ `left` là đầu trái **nhỏ nhất** sao cho `[left..right]` hợp lệ. Khi `right` tăng, `left` chỉ có thể tăng.

**Quy tắc:**
- Thêm `cost[right]` vào `total`.
- **Nếu** `total > maxCost` **thì** bỏ `cost[left]`, `left += 1` — lặp lại (`while`) đến khi hợp lệ.
- Cập nhật `best = max(best, right − left + 1)`.

**Invariant:** sau mỗi bước, `[left..right]` là đoạn hợp lệ dài nhất kết thúc tại `right`.

## 7. Các cách tiếp theo

### Cách 2 — Tổng tiền tố + tìm kiếm nhị phân theo **độ dài**

Tính khả thi đơn điệu: nếu có đoạn dài `L` hợp lệ thì có đoạn dài `L − 1` hợp lệ (bỏ một đầu). ⇒ nhị phân trên `L`, mỗi lần kiểm tra mọi đoạn dài `L` bằng tiền tố trong `O(n)`.

In [9]:
def solve_binary_length(s: str, t: str, maxCost: int) -> int:
    cost = [abs(ord(a) - ord(b)) for a, b in zip(s, t)]
    n = len(cost)
    P = list(accumulate(cost, initial=0))

    def feasible(L):
        return any(P[i + L] - P[i] <= maxCost for i in range(n - L + 1))

    lo, hi = 0, n                  # feasible(0) luôn đúng; tìm L lớn nhất khả thi
    while lo < hi:
        mid = (lo + hi + 1) // 2   # làm tròn lên để tránh lặp vô hạn
        if feasible(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo


for s, t, mc, want in examples:
    assert solve_binary_length(s, t, mc) == want
print([solve_binary_length(*e[:3]) for e in examples])

[3, 1, 1]


**Độ phức tạp:** `O(n log n)` thời gian, `O(n)` bộ nhớ. Chú ý mẫu "tìm **lớn nhất** thỏa": `mid = (lo + hi + 1) // 2`, `lo = mid` / `hi = mid − 1`.

### Cách 3 — Tổng tiền tố + `bisect` theo **đầu trái**

Với mỗi `r`, đầu trái nhỏ nhất `l = bisect_left(P, P[r+1] − maxCost)`; độ dài `r + 1 − l`.

In [10]:
def solve_bisect(s: str, t: str, maxCost: int) -> int:
    P = [0]
    for a, b in zip(s, t):
        P.append(P[-1] + abs(ord(a) - ord(b)))
    best = 0
    for r in range(len(s)):
        l = bisect.bisect_left(P, P[r + 1] - maxCost)
        best = max(best, r + 1 - l)
    return best


for s, t, mc, want in examples:
    assert solve_bisect(s, t, mc) == want
print([solve_bisect(*e[:3]) for e in examples])

[3, 1, 1]


**Độ phức tạp:** `O(n log n)`, bộ nhớ `O(n)`. Nếu `l > r` (tức `cost[r] > maxCost`) thì `r + 1 − l = 0` — tự động đúng.

### Cách 4 — Cửa sổ trượt co giãn (lời giải chuẩn)

In [11]:
def solve_window(s: str, t: str, maxCost: int) -> int:
    left = total = best = 0
    for right in range(len(s)):
        total += abs(ord(s[right]) - ord(t[right]))
        while total > maxCost:
            total -= abs(ord(s[left]) - ord(t[left]))
            left += 1
        best = max(best, right - left + 1)
    return best


for s, t, mc, want in examples:
    assert solve_window(s, t, mc) == want
print([solve_window(*e[:3]) for e in examples])

[3, 1, 1]


**Độ phức tạp:** `O(n)` (mỗi chỉ số vào/ra một lần), bộ nhớ `O(1)`.

### Cách 5 — Cửa sổ "không bao giờ co" (non-shrinking window)

Mẹo: ta chỉ quan tâm độ dài **lớn nhất**, nên cửa sổ không cần co nhỏ hơn kích thước tốt nhất đã đạt. Mỗi bước tăng `right`; nếu quá ngân sách thì **chỉ dịch `left` một bước** (dùng `if` thay `while`) — cửa sổ giữ nguyên kích thước, trượt sang phải. Đáp án là kích thước cuối cùng `n − left`.

In [12]:
def solve_nonshrinking(s: str, t: str, maxCost: int) -> int:
    left = total = 0
    for right in range(len(s)):
        total += abs(ord(s[right]) - ord(t[right]))
        if total > maxCost:
            total -= abs(ord(s[left]) - ord(t[left]))
            left += 1
    return len(s) - left


for s, t, mc, want in examples:
    assert solve_nonshrinking(s, t, mc) == want
print([solve_nonshrinking(*e[:3]) for e in examples])

[3, 1, 1]


**Vì sao đúng?** Kích thước cửa sổ `right − left + 1` không bao giờ giảm: tăng 1 khi hợp lệ, giữ nguyên khi không. Cửa sổ chỉ "lớn lên" khi tồn tại một đoạn hợp lệ dài hơn tất cả trước đó. Lưu ý: cửa sổ lúc đang trượt có thể **không hợp lệ**, nhưng kích thước của nó luôn bằng độ dài tốt nhất đã thấy. **Độ phức tạp:** `O(n)`, `O(1)`.

## 8. TRACE từng bước

Trace Cách 4 trên `s = "abcd"`, `t = "bcdf"` (`cost = [1,1,1,2]`), `maxCost = 3`:

| right | cost | total sau cộng | thu hẹp? | left | cửa sổ | dài | best |
|---|---|---|---|---|---|---|---|
| 0 | 1 | 1 | không | 0 | [0,0] | 1 | 1 |
| 1 | 1 | 2 | không | 0 | [0,1] | 2 | 2 |
| 2 | 1 | 3 | không | 0 | [0,2] | 3 | 3 |
| 3 | 2 | 5 | bỏ cost[0]=1 → 4, bỏ cost[1]=1 → 3 | 2 | [2,3] | 2 | 3 |

In [13]:
def trace_window(s, t, maxCost):
    cost = [abs(ord(a) - ord(b)) for a, b in zip(s, t)]
    print("cost =", cost)
    left = total = best = 0
    for right in range(len(s)):
        total += cost[right]
        removed = []
        while total > maxCost:
            total -= cost[left]; removed.append(left); left += 1
        best = max(best, right - left + 1)
        print(f"right={right} +{cost[right]} bỏ chỉ số {removed or '-'} -> [{left},{right}] total={total} dài={right - left + 1} best={best}")
    return best


trace_window("abcd", "bcdf", 3)
print()
trace_window("krrgw", "zjxss", 19)

cost = [1, 1, 1, 2]
right=0 +1 bỏ chỉ số - -> [0,0] total=1 dài=1 best=1
right=1 +1 bỏ chỉ số - -> [0,1] total=2 dài=2 best=2
right=2 +1 bỏ chỉ số - -> [0,2] total=3 dài=3 best=3
right=3 +2 bỏ chỉ số [0, 1] -> [2,3] total=3 dài=2 best=3

cost = [15, 8, 6, 12, 4]
right=0 +15 bỏ chỉ số - -> [0,0] total=15 dài=1 best=1
right=1 +8 bỏ chỉ số [0] -> [1,1] total=8 dài=1 best=1
right=2 +6 bỏ chỉ số - -> [1,2] total=14 dài=2 best=2
right=3 +12 bỏ chỉ số [1] -> [2,3] total=18 dài=2 best=2
right=4 +4 bỏ chỉ số [2] -> [3,4] total=16 dài=2 best=2


2

In [14]:
def trace_nonshrinking(s, t, maxCost):
    cost = [abs(ord(a) - ord(b)) for a, b in zip(s, t)]
    left = total = 0
    for right in range(len(s)):
        total += cost[right]
        act = "giữ"
        if total > maxCost:
            total -= cost[left]; left += 1; act = "trượt"
        print(f"right={right} [{left},{right}] kích thước={right - left + 1} total={total} ({act})")
    return len(s) - left


print("=>", trace_nonshrinking("abcd", "bcdf", 3))

right=0 [0,0] kích thước=1 total=1 (giữ)
right=1 [0,1] kích thước=2 total=2 (giữ)
right=2 [0,2] kích thước=3 total=3 (giữ)
right=3 [1,3] kích thước=3 total=4 (trượt)
=> 3


## 9. Code nộp LeetCode

In [15]:
class Solution:
    def equalSubstring(self, s: str, t: str, maxCost: int) -> int:
        cost = [abs(ord(a) - ord(b)) for a, b in zip(s, t)]
        left = total = best = 0
        for right, c in enumerate(cost):
            total += c
            while total > maxCost:
                total -= cost[left]
                left += 1
            if right - left + 1 > best:
                best = right - left + 1
        return best


sol = Solution()
for s, t, mc, want in examples:
    print(sol.equalSubstring(s, t, mc))
    assert sol.equalSubstring(s, t, mc) == want

3
1
1


## 10. Vì sao đúng

**Invariant:** sau khi xử lý `right`, `left` là chỉ số **nhỏ nhất** sao cho `sum(cost[left..right]) ≤ maxCost` (hoặc `left = right + 1` nếu `cost[right] > maxCost`, cho độ dài 0).

- **Duy trì:** giả sử đúng với `right − 1` (đầu trái tối thiểu `L`). Với `right`, mọi `l < L` có `sum(cost[l..right−1]) > maxCost` ⇒ `sum(cost[l..right]) ≥` tổng đó `> maxCost` (vì `cost[right] ≥ 0`). Nên đầu trái tối thiểu mới `≥ L` — bắt đầu từ `L` và tăng dần đến khi hợp lệ là đúng.
- **Kết luận:** đoạn tối ưu toàn cục `[l*, r*]` kết thúc tại `r*`; tại bước `right = r*`, `left ≤ l*` nên độ dài ghi nhận `≥ r* − l* + 1`. Vì mọi độ dài ghi nhận đều của đoạn hợp lệ, `best` đúng bằng tối ưu.

Lập luận dựa then chốt vào **`cost ≥ 0`** — với mảng có số âm, cửa sổ trượt không còn đúng.

## 11. Time / Space Complexity

Gọi `n = len(s)`.

| Cách | Time | Space |
|---|---|---|
| Brute `O(n^3)` | `O(n^3)` | `O(1)` |
| 1. Brute có `break` | `O(n^2)` worst | `O(1)` |
| 2. Nhị phân độ dài | `O(n log n)` | `O(n)` |
| 3. Tiền tố + bisect | `O(n log n)` | `O(n)` |
| 4. Cửa sổ co giãn | `O(n)` | `O(1)` (hoặc `O(n)` nếu lưu `cost`) |
| 5. Cửa sổ không co | `O(n)` | `O(1)` |

## 12. Bộ test

In [16]:
solvers = {
    "binary_length": solve_binary_length,
    "bisect": solve_bisect,
    "window": solve_window,
    "nonshrinking": solve_nonshrinking,
    "submit": Solution().equalSubstring,
}
edge = [
    ("a", "a", 0, 1),
    ("a", "z", 0, 0),
    ("a", "z", 25, 1),
    ("az", "za", 24, 0),
    ("abc", "abc", 0, 3),
    ("krrgw", "zjxss", 19, 2),
    ("pxezla", "loewbi", 25, 4),
]
for s, t, mc, want in examples + edge:
    assert brute_raw(s, t, mc) == want, (s, t, mc, brute_raw(s, t, mc))
    for name, f in solvers.items():
        assert f(s, t, mc) == want, (name, s, t, mc, f(s, t, mc), want)
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [17]:
rng = random.Random(1208)
for trial in range(1500):
    n = rng.randint(1, 12)
    alpha = 'abcdefghijklmnopqrstuvwxyz'[: rng.randint(1, 26)]
    s = ''.join(rng.choice(alpha) for _ in range(n))
    t = ''.join(rng.choice(alpha) for _ in range(n))
    mc = rng.randint(0, 40)
    want = brute_raw(s, t, mc)
    assert brute(s, t, mc) == want
    for name, f in solvers.items():
        assert f(s, t, mc) == want, (name, s, t, mc)
print("1500 test ngẫu nhiên khớp brute force")

1500 test ngẫu nhiên khớp brute force


In [18]:
rng = random.Random(3)
n = 10**5
S = ''.join(rng.choice('abcdefghijklmnopqrstuvwxyz') for _ in range(n))
T = ''.join(rng.choice('abcdefghijklmnopqrstuvwxyz') for _ in range(n))
for name, f in solvers.items():
    t0 = time.perf_counter(); r = f(S, T, 10**5); print(f"{name:14s} {r:6} {time.perf_counter() - t0:.3f}s")
assert Solution().equalSubstring(S, S, 0) == n
assert Solution().equalSubstring('a' * n, 'z' * n, 10**6) == 10**6 // 25
print("test giới hạn OK")

binary_length   11709 0.070s


bisect          11709 0.031s


window          11709 0.023s
nonshrinking    11709 0.015s
submit          11709 0.016s


test giới hạn OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — "Reset" cửa sổ khi vượt ngân sách (nhảy `left` tới `right`)

In [19]:
def wrong_reset(s, t, maxCost):
    cost = [abs(ord(a) - ord(b)) for a, b in zip(s, t)]
    left = total = best = 0
    for right, c in enumerate(cost):
        total += c
        if total > maxCost:          # SAI: vứt bỏ cả cửa sổ, chỉ giữ phần tử mới
            left, total = right, c
            if total > maxCost:
                left, total = right + 1, 0
        best = max(best, right - left + 1)
    return best


print(wrong_reset("aaaa", "ccbb", 4), "đúng là", brute_raw("aaaa", "ccbb", 4))

2 đúng là 3


**Vì sao sai:** `cost = [2, 2, 1, 1]`. Khi thêm `cost[2] = 1` tổng thành `5 > 4`, bản sai vứt cả `[0..1]` và bắt đầu lại từ chỉ số 2 — mất đoạn `[1..3]` (tổng `2+1+1 = 4`, dài 3). Phần tử `cost[1]` vẫn có thể thuộc đoạn tối ưu. **Sửa:** chỉ bỏ dần từ bên trái (`while total > maxCost: total -= cost[left]; left += 1`).

**Lưu ý (không phải lỗi):** nhiều bạn thấy lời giải dùng `if` thay cho `while` và nghĩ là sai. Thực ra đó là Cách 5 (cửa sổ không co): kích thước cửa sổ không bao giờ giảm, nên `best` vẫn đúng — dù cửa sổ có lúc **không hợp lệ**. Kiểm chứng:

In [20]:
def if_shrink_with_best(s, t, maxCost):
    left = total = best = 0
    for right in range(len(s)):
        total += abs(ord(s[right]) - ord(t[right]))
        if total > maxCost:
            total -= abs(ord(s[left]) - ord(t[left]))
            left += 1
        best = max(best, right - left + 1)
    return best


rng = random.Random(1)
for _ in range(3000):
    n = rng.randint(1, 7)
    s = ''.join(rng.choice('abz') for _ in range(n)); t = ''.join(rng.choice('abz') for _ in range(n))
    mc = rng.randint(0, 30)
    assert if_shrink_with_best(s, t, mc) == brute_raw(s, t, mc)
print("phiên bản 'if' khớp brute force trên 3000 test — nhưng hãy nhớ invariant khác nhau:")
print("  while -> cửa sổ luôn hợp lệ;  if -> kích thước cửa sổ luôn = tốt nhất đã thấy")

phiên bản 'if' khớp brute force trên 3000 test — nhưng hãy nhớ invariant khác nhau:
  while -> cửa sổ luôn hợp lệ;  if -> kích thước cửa sổ luôn = tốt nhất đã thấy


### Lỗi 2 — Tính độ dài `right − left` (lệch 1)

In [21]:
def wrong_length(s, t, maxCost):
    left = total = best = 0
    for right in range(len(s)):
        total += abs(ord(s[right]) - ord(t[right]))
        while total > maxCost:
            total -= abs(ord(s[left]) - ord(t[left])); left += 1
        best = max(best, right - left)           # SAI: thiếu +1
    return best


print(wrong_length("abcd", "bcdf", 3), "đúng là", 3)

2 đúng là 3


**Sửa:** đoạn đóng `[left, right]` có `right − left + 1` phần tử.

### Lỗi 3 — Quên `abs` (chi phí âm)

In [22]:
def wrong_no_abs(s, t, maxCost):
    left = total = best = 0
    for right in range(len(s)):
        total += ord(s[right]) - ord(t[right])  # SAI: có thể âm
        while total > maxCost:
            total -= ord(s[left]) - ord(t[left]); left += 1
        best = max(best, right - left + 1)
    return best


print(wrong_no_abs("az", "za", 0), "đúng là", brute_raw("az", "za", 0))

2 đúng là 0


**Vì sao sai:** `'z' − 'a' = 25` và `'a' − 'z' = −25` triệt tiêu nhau → tổng `0`, tưởng là miễn phí. **Sửa:** `abs(...)`.

### Lỗi 4 — Áp dụng cửa sổ trượt cho mảng có phần tử âm (biến thể)
Nếu đề đổi thành "đoạn dài nhất có tổng `≤ K`" trên mảng **có số âm**, cửa sổ trượt sai:

In [23]:
def window_longest_sum_at_most(arr, K):
    left = total = best = 0
    for right, x in enumerate(arr):
        total += x
        while left <= right and total > K:
            total -= arr[left]; left += 1
        best = max(best, right - left + 1)
    return best


def brute_longest(arr, K):
    best = 0
    for l in range(len(arr)):
        for r in range(l, len(arr)):
            if sum(arr[l:r + 1]) <= K:
                best = max(best, r - l + 1)
    return best


arr, K = [1, -5], 0
print(window_longest_sum_at_most(arr, K), "đúng là", brute_longest(arr, K))

1 đúng là 2


**Vì sao sai:** với `[1, -5]`, `K = 0`: phần tử `1` một mình vượt ngưỡng nên bị loại ngay, nhưng đoạn `[1, -5]` có tổng `-4 ≤ 0`, dài 2. Với số âm, thêm phần tử có thể làm tổng **giảm**, nên đoạn đã bị loại vì "quá đắt" lại có thể hợp lệ khi kéo dài. Tính đơn điệu biến mất. **Sửa:** dùng tiền tố + cấu trúc khác (ví dụ mảng tiền tố max + bisect, hoặc stack đơn điệu). Bài 1208 an toàn vì `cost ≥ 0`.

## 14. Mẹo & nhận dạng pattern

- **"Đoạn con dài nhất / ngắn nhất thỏa ràng buộc tổng" + phần tử không âm** → cửa sổ trượt.
- **Chuỗi so khớp vị trí với chi phí** → đổi về mảng số trước.
- **Chỉ cần độ dài lớn nhất** → cửa sổ không co (mẹo `if` + `n − left`).
- **Tiền tố không giảm** → bisect.
- Kiểm tra tính đơn điệu trước khi dùng hai con trỏ.

## 15. Biến thể

1. **LeetCode 1004 — Max Consecutive Ones III:** chi phí `1` cho mỗi số `0`, ngân sách `k`.
2. **LeetCode 424 — Longest Repeating Character Replacement:** cửa sổ với điều kiện `len − maxfreq ≤ k`.
3. **LeetCode 209 — Minimum Size Subarray Sum:** ngắn nhất có tổng `≥ target` (co khi thỏa).
4. **LeetCode 2024 — Maximize the Confusion of an Exam:** hai lần cửa sổ.
5. **Chi phí mỗi vị trí cho bởi hàm tùy ý** → cùng khung, chỉ đổi cách tính `cost`.

In [24]:
def longest_ones(nums, k):
    """LeetCode 1004 bằng đúng khung cửa sổ của bài này."""
    left = zeros = best = 0
    for right, x in enumerate(nums):
        zeros += (x == 0)
        while zeros > k:
            zeros -= (nums[left] == 0); left += 1
        best = max(best, right - left + 1)
    return best


print(longest_ones([1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0], 2), longest_ones([0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1], 3))

6 10


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | Mọi đầu trái, mở rộng đến khi vượt | `O(n^2)` | `O(1)` | Oracle |
| 2. Nhị phân độ dài | Khả thi đơn điệu theo `L` | `O(n log n)` | `O(n)` | Khi kiểm tra `L` dễ hơn |
| 3. Tiền tố + bisect | Đầu trái nhỏ nhất cho mỗi `r` | `O(n log n)` | `O(n)` | Cần truy vấn nhiều ngưỡng |
| 4. Cửa sổ co giãn | `while total > maxCost` | `O(n)` | `O(1)` | Nộp bài, dễ chứng minh |
| 5. Cửa sổ không co | `if`, đáp án `n − left` | `O(n)` | `O(1)` | Code ngắn nhất |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** `equalSubstring("abcd", "abcd", 0)` và `equalSubstring("aaaa", "bbbb", 2)`?

**Bài 2 (tính tay):** trace Cách 4 cho `s = "krrgw"`, `t = "zjxss"`, `maxCost = 19`.

**Bài 3 (điền TODO):** hoàn thiện `cost_array(s, t)`.

**Bài 4 (tự viết lại):** trả về luôn **vị trí bắt đầu** của đoạn dài nhất (nếu nhiều đoạn, lấy đoạn bắt đầu sớm nhất).

**Bài 5 (mini challenge):** trả lời **nhiều truy vấn** `maxCost` khác nhau trên cùng `s, t` nhanh hơn `O(n)` mỗi truy vấn sau khi tiền xử lý (gợi ý: với mỗi độ dài `L`, chi phí nhỏ nhất của đoạn dài `L`).

In [25]:
def cost_array(s, t):
    # TODO
    pass


def equal_substring_with_start(s, t, maxCost):
    # TODO: trả về (độ dài, vị trí bắt đầu) ; độ dài 0 thì vị trí -1
    raise NotImplementedError


def build_query_answerer(s, t):
    # TODO: tiền xử lý, trả về hàm query(maxCost) -> độ dài lớn nhất
    raise NotImplementedError

# assert cost_array("abcd", "bcdf") == [1, 1, 1, 2]
# assert equal_substring_with_start("abcd", "bcdf", 3) == (3, 0)

## 18. Gợi ý 3 mức

- **Mức 1:** Đổi về mảng chi phí không âm. Đề thành "đoạn dài nhất có tổng ≤ K".
- **Mức 2:** Hai con trỏ: mở rộng phải, khi quá ngân sách thì thu hẹp trái.
- **Mức 3:** Bài 5: `minCost[L]` = chi phí nhỏ nhất trong các đoạn dài `L` (tính bằng tiền tố, `O(n^2)` hoặc khéo hơn). `minCost` không giảm theo `L` (đoạn dài hơn chứa đoạn ngắn hơn), nên mỗi truy vấn là `bisect_right(minCost, maxCost) − 1`.

## 19. Đáp án tham khảo

**Bài 1:** `4` (không tốn gì) và `2` (mỗi ký tự tốn `1`).

**Bài 2:** `cost = [9, 8, 6, 12, 4]`. right=0: total 9 → [0,0] dài 1; right=1: 17 → [0,1] dài 2; right=2: 23 > 19 → bỏ 9 → 14 → [1,2] dài 2; right=3: 26 → bỏ 8 → 18 → [2,3] dài 2; right=4: 22 → bỏ 6 → 16 → [3,4] dài 2. Đáp án `2`.

In [26]:
assert Solution().equalSubstring("abcd", "abcd", 0) == 4
assert Solution().equalSubstring("aaaa", "bbbb", 2) == 2
assert Solution().equalSubstring("krrgw", "zjxss", 19) == 2


def cost_array(s, t):
    return [abs(ord(a) - ord(b)) for a, b in zip(s, t)]


def equal_substring_with_start(s, t, maxCost):
    cost = cost_array(s, t)
    left = total = 0
    best, start = 0, -1
    for right, c in enumerate(cost):
        total += c
        while total > maxCost:
            total -= cost[left]; left += 1
        if right - left + 1 > best:
            best, start = right - left + 1, left
    return best, start


def build_query_answerer(s, t):
    cost = cost_array(s, t)
    n = len(cost)
    P = list(accumulate(cost, initial=0))
    min_cost = [0] + [min(P[i + L] - P[i] for i in range(n - L + 1)) for L in range(1, n + 1)]
    for L in range(1, n + 1):                     # bảo đảm không giảm (thực ra đã không giảm)
        min_cost[L] = max(min_cost[L], min_cost[L - 1])

    def query(maxCost):
        return bisect.bisect_right(min_cost, maxCost) - 1

    return query


assert cost_array("abcd", "bcdf") == [1, 1, 1, 2]
assert equal_substring_with_start("abcd", "bcdf", 3) == (3, 0)
rng = random.Random(19)
for _ in range(400):
    n = rng.randint(1, 10)
    s = ''.join(rng.choice('abcxyz') for _ in range(n)); t = ''.join(rng.choice('abcxyz') for _ in range(n))
    q = build_query_answerer(s, t)
    for mc in range(0, 60, 7):
        assert q(mc) == brute_raw(s, t, mc)
        L, st = equal_substring_with_start(s, t, mc)
        assert L == brute_raw(s, t, mc)
        if L:
            assert sum(cost_array(s, t)[st:st + L]) <= mc
print("Đáp án bài tập: OK")

Đáp án bài tập: OK


## 20. Template nhớ nhanh + Checklist trước khi submit

```python
left = total = best = 0
for right in range(n):
    total += cost[right]
    while total > K:              # cửa sổ luôn hợp lệ
        total -= cost[left]; left += 1
    best = max(best, right - left + 1)
return best

# hoặc cửa sổ không co:
#   if total > K: total -= cost[left]; left += 1
#   return n - left
```

**Checklist:**
- [ ] Chi phí dùng `abs(ord(a) − ord(b))`.
- [ ] Mảng chi phí **không âm** (điều kiện để cửa sổ trượt đúng).
- [ ] `while` (không phải `if`) nếu muốn cửa sổ luôn hợp lệ.
- [ ] Độ dài `right − left + 1`.
- [ ] `maxCost = 0` và ký tự đầu đắt hơn ngân sách → có thể trả `0`.